In [ ]:
import os
import json
import getpass
import traceback
from huggingface_hub import InferenceClient

# =========================================================
# AUTH
# =========================================================
if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Enter your Hugging Face token: ")

client = InferenceClient(
    api_key=os.environ["HF_TOKEN"],
    provider="auto",
)

MODEL = "Qwen/Qwen2.5-7B-Instruct"

# =========================================================
# PERSONA
# =========================================================
SYSTEM_PROMPT = """You are Nile, a helpful and precise research assistant for
university students working on academic papers and theses.

Your personality:
- Calm, encouraging, and clear, like a knowledgeable senior researcher.
- You explain things simply first, then add technical depth if asked.
- You never make up facts, citations, or numbers. If you do not know
  something and cannot compute or look it up with a tool, say so honestly.
- You prefer using your tools over guessing whenever a task involves
  calculation, unit conversion, or saving information.

Your job:
- Help the student understand concepts, check calculations relevant to
  their research, and keep track of notes or reminders they give you
  during the conversation.
"""

# =========================================================
# TOOLS
# =========================================================
def calculator(expression: str) -> str:
    try:
        result = eval(expression, {"__builtins__": {}})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

def save_note(note: str) -> str:
    with open("nile_notes.txt", "a") as f:
        f.write(note + "\n")
    return f"Saved note: '{note}'"

def read_notes(_: str = "") -> str:
    if not os.path.exists("nile_notes.txt"):
        return "No notes saved yet."
    with open("nile_notes.txt") as f:
        return f.read()

TOOL_FUNCTIONS = {
    "calculator": calculator,
    "save_note": save_note,
    "read_notes": read_notes,
}

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a math expression, for example for stats or scoring formulas",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "A math expression to evaluate"}
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "save_note",
            "description": "Save a short note or reminder for the student to a session log",
            "parameters": {
                "type": "object",
                "properties": {
                    "note": {"type": "string", "description": "The note text to save"}
                },
                "required": ["note"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_notes",
            "description": "Retrieve all notes saved so far in this session",
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

# =========================================================
# AGENT LOOP (with debug prints so failures are visible)
# =========================================================
def run_agent(user_prompt, max_steps=5, debug=True):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    for step in range(max_steps):
        if debug:
            print(f"\n--- Step {step + 1}: calling model ---")

        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=TOOLS_SCHEMA,
            )
        except Exception as e:
            print("ERROR calling the model:")
            traceback.print_exc()
            return None

        msg = response.choices[0].message
        tool_calls = getattr(msg, "tool_calls", None)

        if debug:
            print("Model content:", repr(msg.content))
            print("Tool calls requested:", tool_calls)

        if not tool_calls:
            return msg.content  # final answer

        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": tool_calls,
        })

        for call in tool_calls:
            fn_name = call.function.name
            fn_args = json.loads(call.function.arguments)
            if debug:
                print(f"Calling tool: {fn_name}({fn_args})")

            result = TOOL_FUNCTIONS[fn_name](**fn_args)

            if debug:
                print(f"Tool result: {result}")

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })

    return "Max steps reached without a final answer."


# =========================================================
# RUN
# =========================================================
answer = run_agent(
    "Save a note that I need to double-check the SUS mean calculation, "
    "then calculate the average of these three scores: 68, 71, 74."
)

print("\n=== FINAL ANSWER ===")
print(answer)

Enter your Hugging Face token: ··········

--- Step 1: calling model ---
Model content: None
Tool calls requested: [ChatCompletionOutputToolCall(function=ChatCompletionOutputFunctionDefinition(arguments='{"note":"I need to double-check the SUS mean calculation."}', name='save_note', description=None), id='call_px9851o1fozy9bh4zkt0jo1b', type='function', index=0), ChatCompletionOutputToolCall(function=ChatCompletionOutputFunctionDefinition(arguments='{"expression":"(68 + 71 + 74) / 3"}', name='calculator', description=None), id='call_zxxnoo571ix2zecgg2yaf2eu', type='function', index=1)]
Calling tool: save_note({'note': 'I need to double-check the SUS mean calculation.'})
Tool result: Saved note: 'I need to double-check the SUS mean calculation.'
Calling tool: calculator({'expression': '(68 + 71 + 74) / 3'})
Tool result: 71.0

--- Step 2: calling model ---
Model content: "I've saved the note that you need to double-check the SUS mean calculation.\n\nThe average of the scores 68, 71, and 